# 06 — Train and compare fraud-risk models

This notebook compares Logistic Regression and Random Forest baselines with LightGBM. It also checks whether historical graph features help by comparing LightGBM with and without them.

For local training speed, we keep every fraud case in the training period and take a repeatable 1-in-42 sample of non-fraud cases. Validation and test data keep their natural class balance. The alert threshold is chosen on validation only; the latest test period is used only for final reporting.

In [ ]:
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, f1_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd()
feature_path = project_root / "data" / "processed" / "paysim_graph_features.parquet"
if not feature_path.exists():
    raise FileNotFoundError("Run notebook 05_full_data_features.ipynb first.")
feature_sql = feature_path.resolve().as_posix().replace("'", "''")
db = duckdb.connect(database=":memory:")
db.execute("SET threads = 2")

columns = [
    "step", "type", "amount", "oldbalanceOrg", "oldbalanceDest",
    "prior_outgoing_count", "prior_outgoing_amount",
    "prior_incoming_count", "prior_incoming_amount", "isFraud",
]
column_sql = ", ".join(columns)

print("Loading all fraud cases and a repeatable sample of training negatives...")
train = db.execute(f"""
    SELECT {column_sql}
    FROM read_parquet('{feature_sql}')
    WHERE data_split = 'train'
      AND (
          isFraud = 1 OR
          hash(nameOrig || '|' || nameDest || '|' || CAST(step AS VARCHAR) || '|' || CAST(amount AS VARCHAR)) % 42 = 0
      )
""").df()
validation = db.execute(f"""
    SELECT {column_sql} FROM read_parquet('{feature_sql}') WHERE data_split = 'validation'
""").df()
test = db.execute(f"""
    SELECT {column_sql} FROM read_parquet('{feature_sql}') WHERE data_split = 'test'
""").df()

target = "isFraud"
graph_features = [
    "step", "type", "amount", "oldbalanceOrg", "oldbalanceDest",
    "prior_outgoing_count", "prior_outgoing_amount",
    "prior_incoming_count", "prior_incoming_amount",
]
transaction_features = ["step", "type", "amount", "oldbalanceOrg", "oldbalanceDest"]

print(f"Training sample: {len(train):,} rows; {int(train[target].sum()):,} fraud cases")
print(f"Validation: {len(validation):,} rows; {int(validation[target].sum()):,} fraud cases")
print(f"Test: {len(test):,} rows; {int(test[target].sum()):,} fraud cases")

## Encode transaction type

Models need numbers, so we turn the text transaction type into indicator columns. We keep the same columns for validation and test. Account names, the fraud label, the existing system flag, and balances after the transaction are not model inputs.

In [ ]:
def encode(frame, feature_list, expected_columns=None):
    encoded = pd.get_dummies(frame[feature_list], columns=["type"], dtype="float32")
    if expected_columns is not None:
        encoded = encoded.reindex(columns=expected_columns, fill_value=0)
    return encoded.astype("float32")

X_train = encode(train, graph_features)
graph_columns = X_train.columns.tolist()
X_validation = encode(validation, graph_features, graph_columns)
X_test = encode(test, graph_features, graph_columns)
y_train = train[target].astype("int8")
y_validation = validation[target].astype("int8")
y_test = test[target].astype("int8")
negative_positive_ratio = float((y_train == 0).sum() / (y_train == 1).sum())

print(f"Negative-to-positive ratio after sampling: {negative_positive_ratio:.1f}:1")
print(f"Encoded graph feature matrix: {X_train.shape[0]:,} rows x {X_train.shape[1]} columns")

## Select the alert threshold on validation

The models produce risk scores. We choose a threshold on validation data to maximize F1, then freeze it before evaluating the test period. Precision@K and Recall@K show how useful the highest-ranked cases are for a limited analyst queue.

In [ ]:
def choose_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    if len(thresholds) == 0:
        return 0.5, 0.0
    f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_index = int(np.nanargmax(f1_values))
    return float(thresholds[best_index]), float(f1_values[best_index])

def ranking_at_k(labels, scores, k):
    labels = np.asarray(labels)
    k = min(k, len(labels))
    top = np.argsort(scores)[::-1][:k]
    hits = int(labels[top].sum())
    total_frauds = int(labels.sum())
    return hits / k, hits / total_frauds if total_frauds else 0.0

def measure_test(labels, scores, threshold):
    predicted = (scores >= threshold).astype("int8")
    p100, r100 = ranking_at_k(labels, scores, 100)
    p500, r500 = ranking_at_k(labels, scores, 500)
    p1000, r1000 = ranking_at_k(labels, scores, 1000)
    return {
        "test_pr_auc": average_precision_score(labels, scores),
        "test_roc_auc_secondary": roc_auc_score(labels, scores),
        "test_precision": precision_score(labels, predicted, zero_division=0),
        "test_recall": recall_score(labels, predicted, zero_division=0),
        "test_f1": f1_score(labels, predicted, zero_division=0),
        "precision_at_100": p100, "recall_at_100": r100,
        "precision_at_500": p500, "recall_at_500": r500,
        "precision_at_1000": p1000, "recall_at_1000": r1000,
    }

def make_lightgbm(pos_weight):
    return LGBMClassifier(
        n_estimators=200, learning_rate=0.05, num_leaves=31,
        scale_pos_weight=pos_weight, random_state=42, n_jobs=2, verbosity=-1,
    )

## Train and compare models

Logistic Regression is the simple baseline. Random Forest and LightGBM can learn more complex patterns. Class weighting helps the models pay attention to the rare fraud class in the sampled training data. Validation and test data are not sampled.

In [ ]:
models = {
    "Logistic Regression + graph features": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=300, solver="liblinear", class_weight="balanced", random_state=42),
    ),
    "Random Forest + graph features": RandomForestClassifier(
        n_estimators=100, max_depth=14, min_samples_leaf=10,
        class_weight="balanced_subsample", n_jobs=2, random_state=42,
    ),
    "LightGBM + graph features": make_lightgbm(negative_positive_ratio),
}

fitted_models = {}
thresholds = {}
results = []
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    validation_scores = model.predict_proba(X_validation)[:, 1]
    threshold, validation_f1 = choose_threshold(y_validation, validation_scores)
    test_scores = model.predict_proba(X_test)[:, 1]
    row = {
        "model": name,
        "validation_f1": validation_f1,
        "selected_threshold": threshold,
    }
    row.update(measure_test(y_test, test_scores, threshold))
    results.append(row)
    fitted_models[name] = model
    thresholds[name] = threshold

graph_results = pd.DataFrame(results).sort_values("test_pr_auc", ascending=False)
display(graph_results)

## Ablation: compare graph features

An ablation removes a feature group to see whether it helps. Here the model and evaluation periods stay the same; we remove only the prior sender/receiver activity features.

In [ ]:
tabular_features = ["step", "type", "amount", "oldbalanceOrg", "oldbalanceDest"]
X_train_tab = encode(train, tabular_features)
tab_columns = X_train_tab.columns.tolist()
X_validation_tab = encode(validation, tabular_features, tab_columns)
X_test_tab = encode(test, tabular_features, tab_columns)

tabular_lgbm = make_lightgbm(negative_positive_ratio)
print("Training LightGBM with transaction details only...")
tabular_lgbm.fit(X_train_tab, y_train)
tab_validation_scores = tabular_lgbm.predict_proba(X_validation_tab)[:, 1]
tab_threshold, tab_validation_f1 = choose_threshold(y_validation, tab_validation_scores)
tab_test_scores = tabular_lgbm.predict_proba(X_test_tab)[:, 1]
tabular_result = {
    "model": "LightGBM: transaction details only",
    "validation_f1": tab_validation_f1,
    "selected_threshold": tab_threshold,
}
tabular_result.update(measure_test(y_test, tab_test_scores, tab_threshold))

comparison = pd.concat([graph_results, pd.DataFrame([tabular_result])], ignore_index=True)
comparison = comparison.sort_values("test_pr_auc", ascending=False).reset_index(drop=True)
display(comparison[[
    "model", "validation_f1", "selected_threshold", "test_pr_auc",
    "test_precision", "test_recall", "test_f1",
    "precision_at_100", "recall_at_100",
    "precision_at_500", "recall_at_500",
    "precision_at_1000", "recall_at_1000",
]])

## Save the report and a local model

The metrics CSV and feature-importance CSV are reviewable project outputs. The fitted model stays local and can be recreated by rerunning this notebook.

In [ ]:
reports_dir = project_root / "reports"
models_dir = project_root / "models"
reports_dir.mkdir(exist_ok=True)
models_dir.mkdir(exist_ok=True)
comparison.to_csv(reports_dir / "model_comparison.csv", index=False)

# Save the graph-feature LightGBM model for the API stage.
selected_name = "LightGBM + graph features"
model_path = models_dir / "fraud_risk_model.joblib"
joblib.dump({
    "model": fitted_models[selected_name],
    "encoded_columns": graph_columns,
    "feature_columns": graph_features,
    "threshold": float(thresholds[selected_name]),
}, model_path)

importance = pd.DataFrame({
    "feature": graph_columns,
    "importance": fitted_models[selected_name].feature_importances_,
}).sort_values("importance", ascending=False)
importance.to_csv(reports_dir / "feature_importance.csv", index=False)

print(f"Saved metrics to {reports_dir / 'model_comparison.csv'}")
print(f"Saved feature importance to {reports_dir / 'feature_importance.csv'}")
print(f"Saved local model to {model_path}")
display(importance.head(10))

## How to read the evaluation

- **PR-AUC / average precision** measures how well the model ranks rare fraud cases above normal transactions. It is more informative than accuracy for this imbalanced dataset.
- **Precision** asks: among alerts, what fraction are fraud-labeled? **Recall** asks: among fraud-labeled test transactions, what fraction were found.
- **Precision@K / Recall@K** show how useful the top-ranked cases are when investigators have a limited queue.
- The threshold was selected on validation, not test.
- PaySim is synthetic, and its fraud rate changes substantially over time. These results are a benchmark on this dataset, not a guarantee about real financial systems.